# Demo: Prompt Construction for RAG
### Module 5, Topic 6 — RAG from Scratch

**What you'll see in this notebook:**
1. Run the weak, minimal prompt from Topic 5 and see what's missing
2. Build a stronger prompt: explicit instruction, numbered chunks, citation request
3. Run the same questions again and compare the two versions side by side
4. Update `answer_question()` from Topic 5 to use the improved prompt

Nothing about retrieval changes today — the chunks and the retriever are identical to Topic 5. Only the prompt gets sharper.


## Step 0 — Set Up Both Clients and the Knowledge Base

Identical setup to Topic 5 — same chunks, same embeddings, same retriever.

In [ ]:
!pip install anthropic voyageai --quiet

In [ ]:
import os
import anthropic
import voyageai
import numpy as np

claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

CHAT_MODEL = "claude-3-5-sonnet-20241022"
EMBED_MODEL = "voyage-4"

chunks = [
    "Naija One Bank — Flexi Save Account Policy (Effective 2026)\n\nThe Flexi Save account is Naija One Bank's flagship savings product for individual customers.",
    "It is designed for customers who want easy access to their funds while still earning competitive interest.",
    "The account has no monthly maintenance fee as long as the minimum balance is maintained.",
    "Interest is calculated daily and credited monthly at a rate of 4.2% per annum.",
    "The minimum opening balance required to activate the account is NGN 5,000.",
    "To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.",
    "Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit.",
    "Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.",
    "Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest.",
    "Customers can reactivate Flexi Save status by restoring the minimum balance.",
]

chunk_embeddings = vo.embed(chunks, model=EMBED_MODEL, input_type="document").embeddings
knowledge_base = list(zip(chunks, chunk_embeddings))

def cosine_similarity(vec_a, vec_b):
    vec_a = np.array(vec_a)
    vec_b = np.array(vec_b)
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def retrieve(query, knowledge_base, k=3):
    query_embedding = vo.embed([query], model=EMBED_MODEL, input_type="query").embeddings[0]
    scores = []
    for chunk_text, chunk_vec in knowledge_base:
        score = cosine_similarity(query_embedding, chunk_vec)
        scores.append((chunk_text, score))
    ranked = sorted(scores, key=lambda pair: pair[1], reverse=True)
    return ranked[:k]

print("Knowledge base and retriever ready.")

## Step 1 — Topic 5's Original `build_prompt()`

This is exactly what we used last topic — a working, but minimal, prompt.

In [ ]:
def build_prompt_v1(question, retrieved_chunks):
    context = "\n\n".join(chunk_text for chunk_text, score in retrieved_chunks)

    prompt = f"""Use only the information in the DOCUMENT below to answer the QUESTION.
If the DOCUMENT does not contain enough information to answer, say so clearly instead of guessing.

DOCUMENT:
{context}

QUESTION:
{question}
"""
    return prompt

print("build_prompt_v1() ready (Topic 5's version).")

## Step 2 — Run a Question Through Version 1

Notice: even though this prompt works, the answer has no way to show *which* chunk it actually came from.

In [ ]:
question = "How much can I take out of my account each month before I get charged?"
retrieved = retrieve(question, knowledge_base, k=3)

prompt_v1 = build_prompt_v1(question, retrieved)

response_v1 = claude.messages.create(
    model=CHAT_MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt_v1}]
)

print("--- VERSION 1 ANSWER ---")
print(response_v1.content[0].text)

## Step 3 — Build Version 2: Numbered Chunks + Citation Request

Same three-part shape — instruction, context, question — but the context is now numbered, and the instruction explicitly asks for a source citation.

In [ ]:
def build_prompt_v2(question, retrieved_chunks):
    numbered_context = "\n\n".join(
        f"[Source {i+1}]\n{chunk_text}"
        for i, (chunk_text, score) in enumerate(retrieved_chunks)
    )

    prompt = f"""Use only the information in the SOURCES below to answer the QUESTION.
If the SOURCES do not contain enough information to answer, say so clearly instead of guessing.
After your answer, on a new line, state which source number(s) you used, like this: Source: 2

SOURCES:
{numbered_context}

QUESTION:
{question}
"""
    return prompt

print("build_prompt_v2() ready.")

## Step 4 — Run the Same Question Through Version 2

In [ ]:
prompt_v2 = build_prompt_v2(question, retrieved)

response_v2 = claude.messages.create(
    model=CHAT_MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt_v2}]
)

print("--- VERSION 2 ANSWER ---")
print(response_v2.content[0].text)

## Step 5 — Compare the Two Versions

Look at both printed answers above.

- **Version 1:** likely correct, but with no way to verify which chunk it came from
- **Version 2:** the same correct information, now with a `Source: N` line at the end

Both versions probably got the *content* right — the difference this topic focuses on is **traceability**, not correctness. That distinction matters most exactly when an answer is subtly wrong and someone needs to check why.

## Step 6 — Test the "Doesn't Know" Case With Version 2

The out-of-scope question from Topic 5, run through the improved prompt.

In [ ]:
out_of_scope_question = "What's the interest rate on a Naija One Bank car loan?"
retrieved_2 = retrieve(out_of_scope_question, knowledge_base, k=3)

prompt_v2_oos = build_prompt_v2(out_of_scope_question, retrieved_2)

response_v2_oos = claude.messages.create(
    model=CHAT_MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt_v2_oos}]
)

print(response_v2_oos.content[0].text)

## Step 7 — Check the Result

Even without the Topic 5 relevance-threshold check running here, a well-written instruction should push the model toward admitting it can't answer, rather than stretching irrelevant chunks into a guess. In production, both defences run together: the threshold check from Topic 5 catches the case *before* calling the model at all, and the instruction from this topic is the backup for borderline cases that still make it through.

## Step 8 — Update `answer_question()` to Use the Better Prompt

The only change from Topic 5's version: swap in `build_prompt_v2` and keep the citation format intact.

In [ ]:
RELEVANCE_THRESHOLD = 0.4

def is_relevant_enough(retrieved_chunks, threshold=RELEVANCE_THRESHOLD):
    if not retrieved_chunks:
        return False
    return retrieved_chunks[0][1] >= threshold

def answer_question(question, knowledge_base):
    retrieved_chunks = retrieve(question, knowledge_base, k=3)

    if not is_relevant_enough(retrieved_chunks):
        return "I don't have information about that in the Flexi Save policy document. Please contact Naija One Bank support directly."

    prompt = build_prompt_v2(question, retrieved_chunks)

    response = claude.messages.create(
        model=CHAT_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text

print("answer_question() now uses the improved, citation-aware prompt.")

## Step 9 — Final Check

One call, same pipeline as Topic 5, now with better prompt construction underneath it.

In [ ]:
print(answer_question("How much can I take out of my account each month before I get charged?", knowledge_base))

## What's Next

We now have a full RAG pipeline that retrieves accurately, answers only from what it retrieved, admits when it doesn't know, and cites its source. The last remaining question is: **how do we know if this system is actually performing well, beyond spot-checking a couple of questions by hand?** Topic 7 covers evaluation.